## FineTuning LLM with Model-As-Service

This sample shows how use create a standalone FineTuning job to fine tune a model to summarize a dialog between 2 people using samsum dataset.

#### Training data
We use sample data placed along with the notebook for our finetuning in files "train.jsonl" and "validation.jsonl".

#### Model
We will use the Phi-3-mini-4k-instruct model to show how user can finetune a model for chat-completion task. If you opened this notebook from a specific model card, remember to replace the specific model name. 

#### Outline
1. Setup pre-requisites
2. Pick a model to fine-tune.
3. Create training and validation datasets.
4. Configure the fine tuning job.
5. Submit the fine tuning job.
6. Create serverless deployment using finetuned model and sample inference

### 1. Setup pre-requisites
* Install dependencies
* Connect to AzureML Workspace. Learn more at [set up SDK authentication](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-setup-authentication?tabs=sdk). Replace  `<WORKSPACE_NAME>`, `<RESOURCE_GROUP>` and `<SUBSCRIPTION_ID>` below.
* Connect to `azureml` system registry
* Set an optional experiment name

**Install dependencies by running below cell. This is not an optional step if running in a new environment.**

In [1]:
%pip install azure-ai-ml
%pip install azure-identity

%pip install mlflow
%pip install azureml-mlflow

Note: you may need to restart the kernel to use updated packages.

  Using cached msal_extensions-1.2.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached portalocker-2.10.1-py3-none-any.whl.metadata (8.5 kB)
Using cached msal_extensions-1.2.0-py3-none-any.whl (19 kB)
Using cached portalocker-2.10.1-py3-none-any.whl (18 kB)
  Using cached docker-7.1.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached pyasn1_modules-0.4.1-py3-none-any.whl.metadata (3.5 kB)
  Using cached rsa-4.9-py3-none-any.whl.metadata (4.2 kB)
  Using cached pyasn1-0.6.1-py3-none-any.whl.metadata (8.4 kB)
   ---------------------------------------- 0.0/28.4 MB ? eta -:--:--
   ---------------------------------------  28.3/28.4 MB 224.4 MB/s eta 0:00:01
   ---------------------------------------- 28.4/28.4 MB 112.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/6.0 MB ? eta -:--:--
   ---------------------------------------- 6.0/6.0 

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
azure-storage-file-datalake 12.18.1 requires azure-storage-blob>=12.24.1, but you have azure-storage-blob 12.19.0 which is incompatible.


### Create AzureML Workspace connections

In [10]:
from azure.ai.ml import MLClient
from azure.identity import (
    DefaultAzureCredential,
    InteractiveBrowserCredential,
)

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/cfcbf395-b3e3-41c4-9998-d3e5bbdeaa8d")
except Exception as ex:
    credential = InteractiveBrowserCredential(tenant_id="cfcbf395-b3e3-41c4-9998-d3e5bbdeaa8d")

try:
    workspace_ml_client = MLClient.from_config(credential=credential, tenant_id="cfcbf395-b3e3-41c4-9998-d3e5bbdeaa8d")
except:
    workspace_ml_client = MLClient(
        credential,
        subscription_id="1bd798d8-5940-47f2-81b4-b8389610f09c",
        resource_group_name="rg-srsaggam-1603_ai",
        workspace_name="srsaggam-5838",
    )

# the models, fine tuning pipelines and environments are available in various AzureML system registries,
# Example: Phi family of models are in "azureml", Llama family of models are in "azureml-meta" registry.
registry_ml_client = MLClient(credential, registry_name="azureml")

# Get AzureML workspace object.
workspace = workspace_ml_client._workspaces.get(workspace_ml_client.workspace_name)
workspace.id

Exception in thread Thread-35 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\srsaggam\AppData\Local\anaconda3\envs\ss1\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\srsaggam\AppData\Local\anaconda3\envs\ss1\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\srsaggam\AppData\Local\anaconda3\envs\ss1\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\srsaggam\AppData\Local\anaconda3\envs\ss1\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
  File "c:\Users\srsaggam\AppData\Local\anaconda3\envs\ss1\lib\codecs.py", line 322, in decode
    (result, consumed) = self._buffer_decode(data, self.errors, final)
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa0 in position 69: invalid start byte
DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credential

'/subscriptions/1bd798d8-5940-47f2-81b4-b8389610f09c/resourceGroups/rg-srsaggam-1603_ai/providers/Microsoft.MachineLearningServices/workspaces/srsaggam-5838'

### 2. Pick a model to fine tune

`Phi-3-mini-4k-instruct` is a 3.8B parameters, lightweight, state-of-the-art open model built upon datasets used for Phi-2. The model belongs to the Phi-3 model family, and the Mini version comes in two variants 4K and 128K which is the context length (in tokens) it can support. You can browse these models in the Model Catalog in the Azure AI Studio, filtering by the `chat-completion` task. In this example, we use the `Phi-3-mini-4k-instruct` model. If you have opened this notebook for a different model, replace the model name and version accordingly.

Note the model id property of the model. This will be passed as input to the fine tuning job. This is also available as the `Asset ID` field in model details page in Azure AI Studio Model Catalog.

In [8]:
model_name = "Phi-4-mini-instruct"
model_to_finetune = registry_ml_client.models.get(model_name, label="latest")
print(
    "\n\nUsing model name: {0}, version: {1}, id: {2} for fine tuning".format(
        model_to_finetune.name, model_to_finetune.version, model_to_finetune.id
    )
)



Using model name: Phi-4-mini-instruct, version: 1, id: azureml://registries/azureml/models/Phi-4-mini-instruct/versions/1 for fine tuning


### 3. Sample data
The chat-completion dataset entry using the following schema:


    {
        "prompt": "Create a fully-developed protagonist who is challenged to survive within a dystopian society under the rule of a tyrant. ...",
        "messages":[",
            {",
                "content": "Create a fully-developed protagonist who is challenged to survive within a dystopian society under the rule of a tyrant. ...",
                "role": "user",
            },
            {",
                "content": "Name: Ava\n Ava was just 16 years old when the world as she knew it came crashing down. The government had collapsed, leaving behind a chaotic and lawless society. ...",
                "role": "assistant",
            },
            {",
                "content": "Wow, Ava's story is so intense and inspiring! Can you provide me with more details.  ...",
                "role": "user",
            },
            {
                "content": "Certainly! ....",
                "role": "assistant"",
            }
        ],
        "prompt_id": "d938b65dfe31f05f80eb8572964c6673eddbd68eff3db6bd234d7f1e3b86c2af",
    }

#### Create data inputs

##### Training data

In [19]:
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Data

dataset_version = "1"
train_dataset_name = "chat_training_small"
try:
    train_data_asset = workspace_ml_client.data.get(
        train_dataset_name, version=dataset_version
    )
    print(f"Dataset {train_dataset_name} already exists")
except:
    print("creating dataset")
    train_data = Data(
        path=f"./train.jsonl",
        type=AssetTypes.URI_FILE,
        description="Training dataset",
        name=train_dataset_name,
        version="1",
    )
    train_data_asset = workspace_ml_client.data.create_or_update(train_data)

Dataset chat_training_small already exists


##### Validation data (Optional)

In [20]:
from azure.ai.ml.entities import Data

dataset_version = "1"
validation_dataset_name = "chat_validation_small"
try:
    validation_data_asset = workspace_ml_client.data.get(
        validation_dataset_name, version=dataset_version
    )
    print(f"Dataset {validation_dataset_name} already exists")
except:
    print("creating dataset")
    validation_data = Data(
        path=f"./validation.jsonl",
        type=AssetTypes.URI_FILE,
        description="Validation dataset",
        name=validation_dataset_name,
        version="1",
    )
    validation_data_asset = workspace_ml_client.data.create_or_update(validation_data)

Dataset chat_validation_small already exists


#### Create marketplace subscription for 3P models
**Note:** Skip this step for 1P(Microsoft) models that are offered on Azure. Example: Phi family of models

In [9]:
model_id_to_subscribe = "/".join(model_to_finetune.id.split("/")[:-2])
print(model_id_to_subscribe)

normalized_model_name = model_name.replace(".", "-")

azureml://registries/azureml/models/Phi-4-mini-instruct


In [ ]:
from azure.ai.ml.entities import MarketplaceSubscription


subscription_name = f"{normalized_model_name}-sub"

marketplace_subscription = MarketplaceSubscription(
    model_id=model_id_to_subscribe,
    name=subscription_name,
)

# note: this will throw exception if the subscription already exists or subscription is not required (for example, if the model is not in the marketplace like Phi family)
try:
    marketplace_subscription = (
        workspace_ml_client.marketplace_subscriptions.begin_create_or_update(
            marketplace_subscription
        ).result()
    )
except Exception as ex:
    print(ex)

### 3. Submit the fine tuning job using the the model and data as inputs
 
Create FineTuning job using all the data that we have so far.

#### Define finetune parameters

##### There are following set of parameters that are required.

1. `model` - Base model to finetune.
2. `training_data` - Training data for finetuning the base model.
3. `validation_data` - Validation data for finetuning the base model.
4. `task` - FineTuning task to perform. eg. CHAT_COMPLETION for chat-completion finetuning jobs.
5. `outputs`- Output registered model name.

##### Following parameters are optional:

1. `hyperparameters` - Parameters that control the FineTuning behavior at runtime.
2. `name`- FineTuning job name
3. `experiment_name` - Experiment name for FineTuning job.
4. `display_name` - FineTuning job display name.

In [21]:
from azure.ai.ml.finetuning import FineTuningTaskType, create_finetuning_job
import uuid

guid = uuid.uuid4()
short_guid = str(guid)[:8]
display_name = f"{model_name}-display-name-{short_guid}-from-sdk"
name = f"{model_name}t-{short_guid}-from-sdk"
output_model_name_prefix = f"{model_name}-{short_guid}-from-sdk-finetuned"
experiment_name = f"{model_name}-from-sdk"

finetuning_job = create_finetuning_job(
    task=FineTuningTaskType.CHAT_COMPLETION,
    training_data=train_data_asset.id,
    validation_data=validation_data_asset.id,
    hyperparameters={
        "per_device_train_batch_size": "1",
        "learning_rate": "0.00002",
        "num_train_epochs": "1",
    },
    model=model_to_finetune.id,
    display_name=display_name,
    name=name,
    experiment_name=experiment_name,
    tags={"foo_tag": "bar"},
    properties={"my_property": "my_value"},
    output_model_name_prefix=output_model_name_prefix,
)

In [22]:
created_job = workspace_ml_client.jobs.create_or_update(finetuning_job)
workspace_ml_client.jobs.get(created_job.name)

pathOnCompute is not a known attribute of class <class 'azure.ai.ml._restclient.v2023_04_01_preview.models._models_py3.MLFlowModelJobOutput'> and will be ignored


Experiment,Name,Type,Status,Details Page
Phi-4-mini-instruct-from-sdk,Phi-4-mini-instructt-96c32628-from-sdk,finetuning,Running,Link to Azure Machine Learning studio


#### Wait for the above job to complete successfully

In [23]:
status = workspace_ml_client.jobs.get(created_job.name).status

import time

while True:
    status = workspace_ml_client.jobs.get(created_job.name).status
    print(f"Current job status: {status}")
    if status in ["Failed", "Completed", "Canceled"]:
        print("Job has finished with status: {0}".format(status))
        break
    else:
        print("Job is still running. Checking again in 30 seconds.")
        time.sleep(30)

Current job status: Running
Job is still running. Checking again in 30 seconds.


KeyboardInterrupt: 

In [42]:
finetune_model_name = created_job.outputs["registered_model"]["name"]
finetune_model_name

'Phi-4-mini-instruct-96c32628-from-sdk-finetuned'

In [ ]:
# Deploy the model as a serverless endpoint

endpoint_name = f"{normalized_model_name}-ft-{short_guid}"  # Name must be unique
model_id = f"azureml://locations/{workspace.location}/workspaces/{workspace._workspace_id}/models/Phi-4-mini-instruct-96c32628-from-sdk-finetuned/versions/1"

#### 4. Create serverless endpoint using the finetuned model


##### Option 1: Create serverless endpoint in the same project where it was finetuned

In [ ]:
from azure.ai.ml.entities import ServerlessEndpoint

serverless_endpoint = ServerlessEndpoint(name=endpoint_name, model_id=model_id)

created_endpoint = workspace_ml_client.serverless_endpoints.begin_create_or_update(
    serverless_endpoint
).result()

##### Option 2 : Create Serverless endpoint in a different Region/Subscription/Project

 Scenario: User wants to create deployment in a different project/region/subscription under same tenant and avoid retraining.

* Create MLClient with target Project

In [14]:
# Create Cross region FT deployment client
from azure.ai.ml.entities import ServerlessEndpoint
from azure.ai.ml import MLClient
from azure.identity import (
    DefaultAzureCredential,
    InteractiveBrowserCredential,
)

try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/cfcbf395-b3e3-41c4-9998-d3e5bbdeaa8d")
except Exception as ex:
    credential = InteractiveBrowserCredential(tenant_id="cfcbf395-b3e3-41c4-9998-d3e5bbdeaa8d")

try:
    workspace_ml_client = MLClient.from_config(credential=credential, tenant_id="cfcbf395-b3e3-41c4-9998-d3e5bbdeaa8d")
except:
    workspace_ml_client = MLClient(
        credential,
        subscription_id="1bd798d8-5940-47f2-81b4-b8389610f09c",
        resource_group_name="rg-srsaggamai",
        workspace_name="srsaggam-2637",
    )

workspace = workspace_ml_client._workspaces.get(workspace_ml_client.workspace_name)

Exception in thread Thread-55 (_readerthread):
Traceback (most recent call last):
  File "c:\Users\srsaggam\AppData\Local\anaconda3\envs\ss1\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\srsaggam\AppData\Local\anaconda3\envs\ss1\lib\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "c:\Users\srsaggam\AppData\Local\anaconda3\envs\ss1\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\srsaggam\AppData\Local\anaconda3\envs\ss1\lib\subprocess.py", line 1515, in _readerthread
    buffer.append(fh.read())
  File "c:\Users\srsaggam\AppData\Local\anaconda3\envs\ss1\lib\codecs.py", line 322, in decode
    (result, consumed) = self._buffer_decode(data, self.errors, final)
UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa0 in position 69: invalid start byte
DefaultAzureCredential failed to retrieve a token from the included credentials.
Attempted credential

* Create marketplace subscription in the target project

In [2]:
from azure.ai.ml.entities import MarketplaceSubscription


subscription_name = f"{normalized_model_name}-sub"

marketplace_subscription = MarketplaceSubscription(
    model_id=model_id_to_subscribe,
    name=subscription_name,
)

# note: this will throw exception if the subscription already exists or subscription is not required (for example, if the model is not in the marketplace like Phi family)
try:
    marketplace_subscription = (
        workspace_ml_client.marketplace_subscriptions.begin_create_or_update(
            marketplace_subscription
        ).result()
    )
except Exception as ex:
    print("hello")
    print(str(ex))

NameError: name 'normalized_model_name' is not defined

In [ ]:
workspace_region = workspace.location
model_to_finetune.tags
supported_regions = model_to_finetune.tags['maas-finetuning-deploy-regions']
supported_regions
if workspace_region in supported_regions:
    print(f"Creating endpoint in the region:{workspace_region}")
    serverless_endpoint = ServerlessEndpoint(name=endpoint_name, model_id=model_id)
    created_endpoint = workspace_ml_client.serverless_endpoints.begin_create_or_update(
        serverless_endpoint
    ).result()
else:
    raise ValueError(f"For the model : {model_to_finetune}, the supported regions for deployment are: {supported_regions}")

Class ServerlessEndpoint: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Method serverless_endpoints: This is an experimental method, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Method begin_create_or_update: This is an experimental method, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Creating endpoint in the region:westus3


auth_mode is not a known attribute of class <class 'azure.ai.ml._restclient.v2024_01_01_preview.models._models_py3.ServerlessEndpoint'> and will be ignored


#### 5. Inference on the Finetuned Serverless Endpoint

In [ ]:
endpoint = workspace_ml_client.serverless_endpoints.get(endpoint_name)
endpoint_keys = workspace_ml_client.serverless_endpoints.get_keys(endpoint_name)
auth_key = endpoint_keys.primary_key

In [ ]:
import requests

url = f"{endpoint.scoring_uri}/v1/chat/completions"

payload = {
    "max_tokens": 1024,
    "messages": [
        {
            "content": "This script is great so far. Can you add more dialogue between Amanda and Thierry to build up their chemistry and connection?",
            "role": "user",
        }
    ],
}
headers = {"Content-Type": "application/json", "Authorization": f"{auth_key}"}

response = requests.post(url, json=payload, headers=headers)

response.json()